# 🏁 Day 1 - 7교시 자율 실습: AI 학습 튜터 만들기

> **이 시간은 자율 실습입니다.** 강사가 가르치지 않고, 본인이 직접 1-6교시 부품을 통합해 결과물을 완성합니다.
>
> 📖 사전 학습: `Day1_Lesson7_Guide.md` (결과물 명세·평가 기준·시연 시나리오·도움말)
>
> 🔗 선행 노트북: `Day1_Lesson1` ~ `Day1_Lesson6` 모두 완료

## 🎯 만들 것

**AI 학습 튜터 챗봇** — 학습 스타일별 튜터 + 메모리·스트리밍·Blocks UI

수업이 끝난 뒤에도 본인이 실제로 활용할 수 있는 학습 도우미입니다.

## 📋 진행 순서 (4단계 × 각 약 15분)

```
Step 0. 초기 설정              ← 패키지 설치, API 키
Step 1. 부품 통합              ← 1-6교시 import + 튜터 사전
Step 2. 메모리 체인 구성        ← 4교시 패턴 + 튜터 변수화
Step 3. 🥉 Bronze 완성         ← 메모리 챗봇 동작 확인 (ChatInterface)
Step 4. 🥈 Silver 완성         ← 스트리밍 + Blocks UI + 초기화
Step 5. 🥇 Gold 도전 (선택)    ← 튜터별 분리 + 학습 기록 복원
Step 6. 시연 시나리오 5종 검증
Step 7. 도전 과제 (자유 확장)
```

## ✅ 평가 기준

- 🥉 **Bronze (기본)**: 5명의 튜터 + 메모리 + 답변 스타일 차이
- 🥈 **Silver (권장)**: Bronze + 스트리밍 + 튜터 설명 + 초기화
- 🥇 **Gold (도전)**: Silver + 튜터별 세션 분리 + 학습 기록 복원

> 💡 **막혀도 5분은 스스로 시도**해보고, 그 다음 가이드 문서의 "막힐 때 도움말" 또는 강사 호출.

시작합시다!


---

# Step 0. 초기 설정

## 0-1. 패키지 설치


In [1]:
# 패키지 설치 (Colab/로컬 공통)
%pip install -q langchain langchain-openai python-dotenv gradio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 0-2. OpenAI API 키 설정


In [4]:
import os
import json
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
import gradio as gr

MODEL_NAME = "gpt-4o-mini"
LOG_PATH = Path("logs/chat_log.json")

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    load_dotenv(Path.cwd() / "langchain-web-chatbot" / ".env")

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("프로젝트 .env에 OPENAI_API_KEY를 설정하세요.")

print(f"API 키 로드 완료 / 모델: {MODEL_NAME}")


c:\workspace\chatbot\langchain-web-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


API 키 로드 완료 / 모델: gpt-4o-mini
✅ import 완료
langchain_openai (Gradio: 6.29.0)


---

# Step 1. 부품 통합 (약 15분)

## 1-1. 필요한 import

| 출처 | 가져올 도구 |
| --- | --- |
| 2교시 (LangChain) | `ChatOpenAI`, `ChatPromptTemplate`, `MessagesPlaceholder` |
| 4교시 (메모리) | `RunnableWithMessageHistory`, `InMemoryChatMessageHistory` |
| 5교시 (Gradio) | `gradio as gr` |


In [8]:
# 1-6교시 부품 통합 import
from pathlib import Path
from datetime import datetime
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
import gradio as gr

print(f"✅ import 완료\nlangchain_openai \nGradio: {gr.__version__}")

✅ import 완료
langchain_openai 
Gradio: 6.29.0


## 1-2. 튜터 사전 만들기

학습 스타일이 명확히 다른 5명의 튜터를 정의합니다. 시스템 메시지 한 줄 차이로 답변이 완전히 달라져요 (1교시 5부).

> 💡 마지막 한두 명은 본인 취향으로 자유롭게 바꿔보세요 (예: "친구처럼 반말 튜터", "한 줄로만 답하는 튜터" 등).


In [10]:
# 튜터 사전: 이름 → system 메시지
TUTORS = {
    "친절한 입문자 튜터":
        "당신은 친절한 입문자 전용 학습 튜터입니다. "
        "어려운 개념도 일상적인 비유와 예시로 쉽게 설명하세요. "
        "전문 용어가 나오면 풀어서 말하고, 학습자가 이해했는지 확인 질문을 한 번씩 던지세요.",

    "코드 실습 튜터":
        "당신은 프로그래밍·데이터 실습 튜터입니다. "
        "모든 답변에 실행 가능한 짧은 Python 코드 예제를 포함하세요. "
        "이론보다 직접 해보는 방식을 우선하며, 코드는 주석과 함께 제공하세요.",

    "이론 정밀 튜터":
        "당신은 학술적 정확성을 중시하는 이론 튜터입니다. "
        "정의를 정확하게 제시하고, 개념 사이의 관계를 명확히 구분해서 설명하세요. "
        "모호한 표현이나 과한 비유는 자제하세요.",

    "시험 대비 튜터":
        "당신은 시험 합격을 돕는 튜터입니다. "
        "핵심 요점을 번호 매겨 정리하고, 자주 출제되는 함정 포인트와 예상 문제를 함께 제시하세요.",

    "소크라테스식 튜터":
        "당신은 답을 바로 알려주지 않는 튜터입니다. "
        "학습자가 스스로 답에 도달하도록 단계별 질문을 던지며 안내하세요. "
        "다만 학습자가 막혀 보이면 적절한 힌트를 주세요.",

    "훈련교관식 튜터":
        "당신은 빡센 특수부대 훈련교관식 튜터입니다."
        "학습자가 답을 틀리면 바로 피드백을 주고면서, 거친 말과 함께 다그치듯 학습시킵니다."
        "정확한 답을 제시하며 반복 학습을 유도하세요. "
}

print(f"✅ {len(TUTORS)}명의 튜터 등록: {list(TUTORS.keys())}")

✅ 6명의 튜터 등록: ['친절한 입문자 튜터', '코드 실습 튜터', '이론 정밀 튜터', '시험 대비 튜터', '소크라테스식 튜터', '훈련교관식 튜터']


---

# Step 2. 메모리 체인 구성 (약 15분)

4교시 `RunnableWithMessageHistory` 패턴을 그대로 가져와, **system 메시지에 튜터 변수만 추가**합니다.

## 2-1. 프롬프트와 LLM 정의

핵심은 system 메시지에 `{tutor_description}` 변수가 들어간다는 점입니다. 호출할 때마다 다른 튜터를 넘길 수 있어요.


In [11]:
# LLM 정의
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 프롬프트: 튜터 변수 + 이전 대화 자리 + 학습자 질문
prompt = ChatPromptTemplate.from_messages([
    ("system", "{tutor_description}"),       # 튜터가 변수
    MessagesPlaceholder("history"),          # 이전 대화 자리 (4교시)
    ("human", "{question}")
])

# 기본 체인
chain = prompt | llm

print("✅ 체인 구성 완료")

✅ 체인 구성 완료


## 2-2. 세션 저장소 함수

4교시와 동일. **session_id마다 별개의 학습 기록**이 생성됩니다.


In [12]:
# 모든 세션의 학습 기록을 담는 사전
store = {}

def get_session_history(session_id: str):
    """session_id에 해당하는 히스토리를 가져오거나 새로 만들기"""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

print("✅ 세션 저장소 준비 완료")

✅ 세션 저장소 준비 완료


## 2-3. 메모리 체인 만들기

`RunnableWithMessageHistory`로 체인을 감쌉니다. 두 키 이름이 정확히 일치해야 메모리가 동작해요 (4교시 4부).


In [14]:
# 메모리 체인 (4교시 4부 패턴 그대로)
# 메모리 wrapper로 감싸기
chain_with_memory = RunnableWithMessageHistory(
    chain,                              # 감쌀 체인
    get_session_history,                # 히스토리 가져올 함수
    input_messages_key="question",      # 사용자 입력 변수 이름 ({question})
    history_messages_key="history",     # MessagesPlaceholder 이름
)

print("✅ 메모리 체인 구성 완료")

✅ 메모리 체인 구성 완료


c:\workspace\chatbot\langchain-web-chatbot\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 2-4. 빠른 동작 확인

UI를 만들기 전에 체인이 잘 동작하는지 먼저 확인합시다. **학습 메모리 유지**가 가장 중요해요.


In [17]:
# 1차: 학습 주제 알려주기
result1 = chain_with_memory.invoke(
    {"question": "Python 리스트가 뭐야? 한 줄로만 설명해줘",
     "tutor_description": TUTORS["훈련교관식 튜터"]},
    config={"configurable": {"session_id": "test"}}
)
print("[1턴]", result1.content)

# 2차: 이전 학습을 기억하는지 확인
result2 = chain_with_memory.invoke(
    {"question": "방금 설명한 거 짧은 예제로 보여줘",
     "tutor_description": TUTORS["훈련교관식 튜터"]},
    config={"configurable": {"session_id": "test"}}
)
print("\n[2턴]", result2.content)

[1턴] 리스트는 여러 값을 순서대로 저장할 수 있는 가변적인 데이터 구조야! 이걸 또 반복하냐? 이제 리스트의 인덱스 접근 방법을 설명해봐! 제대로 이해했는지 확인할 거니까 준비해!

[2턴] 아니, 또 예제를 못 보여주겠다고? 기본적인 거부터 확실히 해! 자, 이걸 보라고!

```python
my_list = [10, 20, 30, 40, 50]
print(my_list[0])  # 10 출력
print(my_list[2])  # 30 출력
```

여기서 `my_list[0]`은 리스트의 첫 번째 요소를 접근하는 거야. 이해했어? 자, 그럼 너도 해봐! 리스트를 만들고 인덱스를 사용해봐! 빨리!


**✅ 결과 확인**

2턴 답변이 **리스트와 관련된 예제**여야 합니다. "무슨 얘긴지 모르겠어요" 같은 답이 나오면:
- 두 호출의 `session_id`가 같은지 확인
- `input_messages_key`, `history_messages_key` 값 확인

→ 가이드 문서 5부 "막힘 2" 참고


---

# Step 3. 🥉 Bronze 완성 (약 10분)

가장 단순한 형태로 챗봇 UI를 띄워봅시다. **5교시에서 배운 `ChatInterface`** 활용.

## 3-1. 튜터 챗봇 함수 정의 (스트리밍 없음)

`gr.ChatInterface`가 자동 호출할 함수입니다. 시그니처는 `(message, history)` 고정.

> 💡 튜터는 일단 "친절한 입문자 튜터" 하나로 고정. 튜터 선택은 Silver에서 추가.


In [18]:
# Bronze 챗봇 함수
def chat_bronze(message, history):
    result = chain_with_memory.invoke(
        {"question": message,
         "tutor_description": TUTORS["친절한 입문자 튜터"]},
        config={"configurable": {"session_id": "bronze"}}
    )
    return result.content

## 3-2. ChatInterface로 띄우기


In [20]:
# 이전 Gradio 인스턴스 종료
gr.close_all()

# Bronze 챗봇 실행
demo_bronze = gr.ChatInterface(
    fn=chat_bronze,
    title="🥉 Bronze AI 학습 튜터",
    description="친절한 입문자 튜터 + 메모리"
)
demo_bronze.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


**✅ Bronze 완성 검증**

브라우저에서 다음을 시도해보세요.

```
1. "Python의 for 반복문 알려줘"     → 친절한 비유로 설명
2. "방금 설명한 거 예제 더 보여줘"  → 이전 맥락 이어가며 예제
3. "지금까지 우리가 무슨 얘기 했지?"  → 학습 내용 요약
```

3개 모두 동작하면 **시나리오 1·2 (Bronze) 통과** 🎉

다음으로 넘어가기 전에 챗봇을 종료시키세요.


In [21]:
# Bronze 챗봇 종료
gr.close_all()
print("✅ Bronze 챗봇 종료")

Closing server running on port: 7862
✅ Bronze 챗봇 종료


---

# Step 4. 🥈 Silver 완성 (약 15분)

Bronze에 다음 3가지를 추가합니다.

- 응답 스트리밍 (한 글자씩 점진 출력)
- 튜터 드롭다운 + 설명 자동 갱신
- 학습 대화 초기화 버튼

## 4-1. 스트리밍 함수 정의

6교시 4부에서 배운 yield 패턴 그대로. **핵심은 누적값을 yield 하는 것**.

| 잘못된 패턴 | 올바른 패턴 |
| --- | --- |
| `yield chunk.content` | `yield full_response` |
| (마지막 chunk만 표시됨) | (점점 길어지는 답변 표시됨) |


In [22]:
# Silver 스트리밍 함수 (Blocks용)
def respond_stream(message, history, tutor_name):
    if not message:
        yield "", history
        return

    # 사용자 메시지를 history에 즉시 추가
    history = history + [{"role": "user", "content": message}]
    history = history + [{"role": "assistant", "content": ""}]

    # 스트리밍 호출
    full_response = ""
    for chunk in chain_with_memory.stream(
        {"question": message,
         "tutor_description": TUTORS[tutor_name]},
        config={"configurable": {"session_id": "silver"}}
    ):
        # chunk.content를 누적하고, history의 마지막 메시지를 갱신
        full_response += chunk.content
        history[-1]["content"] = full_response
        yield "", history                # 입력창 비우기 + 채팅창 갱신

## 4-2. Gradio Blocks UI 구성 + 이벤트 핸들러

좌측(튜터 드롭다운·설명·초기화) + 우측(채팅창) 2열 레이아웃을 한 번에 정의합니다.

> 💡 Gradio 표준 패턴: `with gr.Blocks() as demo:` **하나의 블록 안**에 컴포넌트와 이벤트 핸들러를 모두 정의 (5교시 4-5부).


In [24]:
# Silver 챗봇 UI (Blocks 전체)
gr.close_all()

def clear_chat():
    """학습 대화 초기화: 메모리도 함께 비움"""
    if "silver" in store:
        store["silver"].clear()
    return []

with gr.Blocks(title="AI 학습 튜터") as demo_silver:
    gr.Markdown("# 📚 AI 학습 튜터 (Silver)")
    gr.Markdown("학습 스타일을 골라 질문해보세요. 답변이 한 글자씩 나타납니다.")

    with gr.Row():
        # 좌측 패널: 튜터 선택 + 설명 + 초기화
        with gr.Column(scale=1):
            tutor_dd = gr.Dropdown(choices=list(TUTORS.keys()),
                                    value="친절한 입문자 튜터",
                                    label="튜터 스타일")
            tutor_desc = gr.Textbox(value=TUTORS["친절한 입문자 튜터"],
                                     interactive=False, lines=5,
                                     label="튜터 설명")
            clear_btn = gr.Button("🧹 학습 대화 초기화")

        # 우측 패널: 채팅창
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=420)
            with gr.Row():
                msg = gr.Textbox(placeholder="질문 입력...", scale=4, container=False)
                send = gr.Button("전송", variant="primary", scale=1)

    # 이벤트 핸들러 (같은 with 블록 안에서 정의)
    tutor_dd.change(fn=lambda t: TUTORS[t], inputs=tutor_dd, outputs=tutor_desc)
    send.click(respond_stream, [msg, chatbot, tutor_dd], [msg, chatbot])
    msg.submit(respond_stream, [msg, chatbot, tutor_dd], [msg, chatbot])
    clear_btn.click(fn=clear_chat, outputs=chatbot)

demo_silver.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


**✅ Silver 완성 검증**

다음 3가지를 확인하세요.

```
1. 답변이 한 글자씩 점진적으로 나타나는지 (스트리밍)
2. 튜터 드롭다운을 바꾸면 좌측 설명 텍스트가 즉시 바뀌는지
3. "학습 대화 초기화" 클릭 후 이전 학습 질문 → 모른다고 답하는지
```

3개 모두 OK면 **🥈 Silver 완성!** 시나리오 1, 2, 4, 5 통과.

추가로 시도해볼 만한 것:
- 같은 질문을 5명의 튜터에게 → 답변 스타일이 명확히 다른지 (시나리오 4)

다음으로 넘어가기 전에 챗봇을 종료시키세요.


In [25]:
gr.close_all()
print("✅ Silver 챗봇 종료")

Closing server running on port: 7862
✅ Silver 챗봇 종료


---

# Step 5. 🥇 Gold 도전 (선택, 약 10분)

Silver를 끝낸 학생만 시도하세요. 시간이 부족하면 Step 6 시연 검증으로 건너뛰세요.

## 5-1. 튜터별 session_id 분리

지금까지 모든 튜터가 `session_id="silver"` 하나를 공유하고 있어서, 튜터를 바꿔도 같은 기억을 공유합니다.

**튜터별로 다른 session_id**를 쓰면 각 튜터가 독립된 학습 흐름을 가집니다.

> 💡 4교시 FAQ Q5: `session_id = f"tutor_{name}"` 패턴.


In [26]:
# Gold 스트리밍 함수 (튜터별 분리)
def respond_stream_gold(message, history, tutor_name):
    if not message:
        yield "", history
        return

    history = history + [{"role": "user", "content": message}]
    history = history + [{"role": "assistant", "content": ""}]

    # 튜터 이름을 포함한 session_id
    session_id = f"tutor_{tutor_name}"

    full_response = ""
    for chunk in chain_with_memory.stream(
        {"question": message,
         "tutor_description": TUTORS[tutor_name]},
        config={"configurable": {"session_id": session_id}}
    ):
        full_response += chunk.content
        history[-1]["content"] = full_response
        yield "", history

## 5-2. Gold UI: 튜터 변경 시 학습 기록 복원

튜터가 바뀌면 그 튜터의 기존 학습 기록을 자동으로 불러옵니다.


In [28]:
# Gold UI: 튜터별 학습 기록 자동 복원
gr.close_all()

def load_tutor_history(tutor_name):
    """튜터 선택 시 그 튜터의 기존 메시지를 Gradio 형식으로 변환"""
    session_id = f"tutor_{tutor_name}"
    if session_id not in store:
        return TUTORS[tutor_name], []
    messages = []
    for m in store[session_id].messages:
        role = "user" if m.type == "human" else "assistant"
        messages.append({"role": role, "content": m.content})
    return TUTORS[tutor_name], messages

with gr.Blocks(title="AI 학습 튜터 Gold") as demo_gold:
    gr.Markdown("# 🥇 AI 학습 튜터 (Gold)")
    gr.Markdown("튜터마다 별개의 학습 기록을 유지합니다.")

    with gr.Row():
        with gr.Column(scale=1):
            tutor_dd_g = gr.Dropdown(choices=list(TUTORS.keys()),
                                      value="친절한 입문자 튜터", label="튜터 스타일")
            tutor_desc_g = gr.Textbox(value=TUTORS["친절한 입문자 튜터"],
                                       interactive=False, lines=5, label="튜터 설명")
            clear_btn_g = gr.Button("🧹 현재 튜터 학습 기록 초기화")

        with gr.Column(scale=2):
            chatbot_g = gr.Chatbot(height=420)
            with gr.Row():
                msg_g = gr.Textbox(placeholder="질문 입력...", scale=4, container=False)
                send_g = gr.Button("전송", variant="primary", scale=1)

    # 튜터 변경 → 설명 + 학습 기록 모두 갱신
    tutor_dd_g.change(fn=load_tutor_history, inputs=tutor_dd_g,
                       outputs=[tutor_desc_g, chatbot_g])

    # 전송 / 엔터
    send_g.click(respond_stream_gold, [msg_g, chatbot_g, tutor_dd_g], [msg_g, chatbot_g])
    msg_g.submit(respond_stream_gold, [msg_g, chatbot_g, tutor_dd_g], [msg_g, chatbot_g])

    # 초기화 (현재 튜터만)
    def clear_current(tutor_name):
        sid = f"tutor_{tutor_name}"
        if sid in store:
            store[sid].clear()
        return []
    clear_btn_g.click(fn=clear_current, inputs=tutor_dd_g, outputs=chatbot_g)

demo_gold.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


**✅ Gold 완성 검증 (시나리오 3)**

```
1. "친절한 입문자 튜터" 선택 → "Python 리스트 알려줘"
2. "코드 실습 튜터" 선택 → "지금까지 무슨 얘기 했지?"
   → 모른다고 답함 (다른 session)
3. 다시 "친절한 입문자 튜터" 선택
   → 자동으로 이전 학습 기록 복원!
4. "지금까지 무슨 얘기 했지?" → 리스트 학습 기억함
```

이게 동작하면 **🥇 Gold 완성!**

다음으로 넘어가기 전에 챗봇을 종료시키세요.


In [29]:
gr.close_all()
print("✅ Gold 챗봇 종료")

Closing server running on port: 7862
✅ Gold 챗봇 종료


---

# Step 6. 🎬 시연 시나리오 5종 체크리스트

본인이 도달한 단계의 챗봇을 다시 띄우고 (Silver 또는 Gold), 아래 시나리오를 모두 통과시키세요.

## 체크리스트

```
[ ] 시나리오 1: 기본 동작 (Bronze)
    "Python 리스트 알려줘" → 튜터 스타일로 답변
    답변이 점진 출력됨 (Silver)

[ ] 시나리오 2: 학습 메모리 (Bronze)
    질문 후 "방금 설명한 거 예제 보여줘" → 이전 맥락 이어감

[ ] 시나리오 3: 튜터별 분리 (Gold)
    튜터 바꿔서 "지금까지 무슨 얘기 했지?" → 다른 튜터는 모름

[ ] 시나리오 4: 튜터별 답변 차이 (Bronze)
    같은 질문을 5명에게 → 답변 스타일이 명확히 다름
    - 친절한 입문자 튜터: 비유로 쉽게
    - 코드 실습 튜터: 코드 위주
    - 이론 정밀 튜터: 정확한 정의
    - 시험 대비 튜터: 핵심 요점·예상 문제
    - 소크라테스식 튜터: 답 대신 질문

[ ] 시나리오 5: 초기화 (Silver)
    "학습 대화 초기화" 후 이전 질문 → 모른다고 답함
```

본인 결과물의 등급:
- Bronze: 1, 2, 4 통과
- Silver: 1, 2, 4, 5 통과 + 스트리밍 + 튜터 설명 자동 갱신
- Gold: 1, 2, 3, 4, 5 모두 통과


---

# Step 7. 🎯 도전 과제 (시간 남는 학생용)

본인 챗봇이 시연 시나리오를 모두 통과했다면, 다음 중 하나를 추가해보세요. **실무에서 학습 봇에 자주 추가되는** 기능들입니다.

## 과제 A: 학습 기록 텍스트로 저장 (실무 빈도 ★★★★★)

학습이 끝났을 때 그날의 학습 내용을 텍스트 파일로 저장하면 복습에 매우 유용합니다.

**힌트**
```python
from datetime import datetime

def save_log():
    """현재 silver 세션 메모리를 텍스트로 저장"""
    if "silver" not in store:
        return "저장할 학습 내용이 없습니다."
    lines = []
    for m in store["silver"].messages:
        role = "학습자" if m.type == "human" else "튜터"
        lines.append(f"[{role}] {m.content}")
    filename = f"learning_{datetime.now().strftime('%Y%m%d_%H%M')}.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n\n".join(lines))
    return f"✅ {filename}에 저장됨"
```

저장 버튼(`gr.Button`)을 추가하고 클릭 시 이 함수를 호출하세요.

---

## 과제 B: 튜터별 temperature 다르게 (실무 빈도 ★★★★☆)

1교시에서 배운 temperature가 튜터의 성격과도 연결됩니다.

- **이론 정밀 튜터**: `temperature=0` (정확하고 일관된 답변)
- **소크라테스식 튜터**: `temperature=0.8` (창의적인 질문 유도)

**힌트**: `TUTORS` 사전을 `{name: {"system": ..., "temperature": ...}}` 구조로 확장하고, 호출 시점에 `llm.bind(temperature=...)` 활용.

---

## 과제 C: 튜터를 JSON 파일로 분리 (실무 빈도 ★★★★☆)

튜터가 늘어나면 코드에서 분리해 외부 파일로 관리합니다.

```python
import json
with open("tutors.json", encoding="utf-8") as f:
    TUTORS = json.load(f)
```

비개발자(다른 교사 등)도 튜터를 편집할 수 있게 됩니다.

---

## 과제 D: 학습 주제 표시 (실무 빈도 ★★★☆☆)

학습자가 현재 어떤 주제를 학습 중인지 한 줄로 보여주면 흐름이 명확합니다.

**힌트**: `gr.Textbox(label="현재 학습 주제", interactive=False)` 컴포넌트를 추가하고, 학습자가 새 메시지를 보낼 때마다 LLM에 "지금까지 대화의 주제를 한 단어로 요약" 같은 추가 호출.

---

## 과제 E: 나만의 튜터 만들기 (실무 빈도 ★★★☆☆)

좌측 패널의 "튜터 설명" 텍스트박스를 `interactive=True`로 바꾸고, 편집한 내용이 실제 system 메시지에 반영되게 만드세요.

학생이 자신만의 튜터(예: "코딩 시험 D-7 마지막 정리 튜터")를 직접 만들어 쓸 수 있게 됩니다.

---

> 💡 **5개 다 할 필요 없습니다.** 1개만 골라 깊이 마무리하세요.


---

# 🎓 Day 1 마무리

## 오늘 만든 것

```
1교시 → 6교시 → 7교시
       ↓
LLM 입문에서 시작해, 본인이 직접 쓸 수 있는 학습 도구까지.
```

이 결과물은 단순한 예제가 아니에요. **수업 끝나고도 실제로 쓸 수 있는** 학습 도우미입니다. Python 공부, 통계 공부, 영어 공부 등 어떤 주제에도 활용 가능해요.

같은 패턴을 확장하면 회사·학교의 신입 교육 봇, 코딩 멘토, 외국어 튜터 같은 실무 시스템으로 발전시킬 수 있습니다.

## 다음 시간 (8교시)

- 짝꿍에게 본인 챗봇 시연
- 5종 시나리오 통과 확인
- 서로 코드 리뷰 + 개선 아이디어 공유

본인 결과물에 자신감을 가지고 가세요!

## Day 2 예고

내일은 챗봇에 **도구(Tool)** 를 쥐여주는 법을 배웁니다. 환율 조회, 계산, 외부 정보 검색 같은 작업을 LLM이 스스로 골라 호출하게 만들어요. 오늘 만든 챗봇이 더 강력해집니다.

수고하셨습니다! 🎉
